In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import pandas as pd
import spacy
from spacy import displacy
import stanza

# ================================
# CSV'den Veri Okuma
# ================================
csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv"
df = pd.read_csv(csv_path, engine="python")

# Eğer 'processed_final_review' sütunu varsa onu kullanın,
# aksi halde 'final_review' sütunundan çalışabilirsiniz.
if "processed_final_review" in df.columns:
    reviews = df["processed_final_review"].fillna("").tolist()
else:
    reviews = df["final_review"].fillna("").tolist()

# Örnek olarak boş olmayan ilk yorumu seçelim.
sample_review = None
for review in reviews:
    if review.strip():
        sample_review = review.strip()
        break

if not sample_review:
    print("Veride işlenecek bir yorum bulunamadı.")
else:
    print("=== Örnek Yorum ===")
    print(sample_review)
    print("\n====================\n")

# ================================
# Dependency Parsing (Spacy ile)
# ================================
print(">>> Spacy ile Dependency Parsing:")

# Spacy 'en_core_web_sm' modelini yüklüyoruz (model yoksa 'python -m spacy download en_core_web_sm' komutuyla indiriniz)
nlp_spacy = spacy.load("en_core_web_sm")
doc_spacy = nlp_spacy(sample_review)

# Her token için: token.text, bağımlılık etiketi, ve head token'ı yazdırıyoruz.
for token in doc_spacy:
    print(f"{token.text:15} {token.dep_:10} {token.head.text}")

# Görselleştirme (jupyter ortamında çalışıyorsa, aşağıdaki kod dependency grafiğini render eder)
displacy.render(doc_spacy, style="dep", jupyter=True)

# ================================
# Constituency Parsing (Stanza ile)
# ================================
print("\n>>> Stanza ile Constituency Parsing:")

# Stanza pipeline'ı kuruyoruz (ilk kullanımda modeli indirmeniz gerekebilir)
stanza.download('en', processors='tokenize,pos,constituency', verbose=False)
nlp_stanza = stanza.Pipeline('en', processors='tokenize,pos,constituency', verbose=False)
doc_stanza = nlp_stanza(sample_review)

# Her cümlenin constituency ağacını yazdırıyoruz
for i, sentence in enumerate(doc_stanza.sentences):
    print(f"\nCümle {i+1} Constituency Ağacı:")
    print(sentence.constituency)



=== Örnek Yorum ===
stay cheap narrow room know needed room could spend night look hotel cheap room enough room really small bed really small except mini fridge want open large suitcase not open bathroom toilet enough room give night provide with basic condition stay bed pillow comfortable toilet bathroom clean stay night want stay second night room could catch eye night tolerate internet not work room wifi password request location term walk distance everywhere


>>> Spacy ile Dependency Parsing:
stay            ROOT       stay
cheap           amod       room
narrow          amod       room
room            compound   room
know            nmod       room
needed          amod       room
room            nsubj      spend
could           aux        spend
spend           ccomp      stay
night           dobj       spend
look            xcomp      spend
hotel           nmod       room
cheap           amod       room
room            nmod       room
enough          amod       room
room         


>>> Stanza ile Constituency Parsing:

Cümle 1 Constituency Ağacı:
(ROOT (S (VP (VB stay) (SBAR (S (NP (JJ cheap) (JJ narrow) (NN room) (VB know) (NP (VBN needed) (NN room))) (VP (MD could) (VP (VB spend) (NP (NN night)) (NP (NP (ADJP (VP (VB look) (NP (NN hotel) (JJ cheap) (NN room))) (JJ enough)) (NN room)) (NP (NP (ADJP (RB really) (JJ small)) (NN bed)) (ADJP (RB really) (JJ small) (SBAR (IN except) (S (NP (NN mini) (NN fridge)) (VP (VBP want) (S (NP (JJ open) (JJ large) (NN suitcase)) (RB not) (JJ open) (NN bathroom) (NN toilet) (JJ enough) (NP (NP (NN room)) (S (VP (VB give) (NP (NN night)) (S (VP (VB provide) (PP (IN with) (NP (NML (JJ basic) (NN condition) (NN stay)) (NN bed) (NN pillow) (JJ comfortable) (NN toilet) (NN bathroom) (NML (JJ clean) (NN stay)) (NN night)))))) (VP (VB want) (VP (VB stay) (SBAR (S (NP (NML (JJ second) (NN night)) (NN room)) (VP (MD could) (VP (VB catch) (NP (NN eye) (NN night)) (VP (VB tolerate) (S (NP (NN internet)) (RB not) (NN work) (NML (NML (NML 

In [11]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import pandas as pd
import spacy
from spacy import displacy
import stanza
from nltk import Tree
from IPython.display import display
from graphviz import Digraph

# ================================
# CSV'den Veri Okuma
# ================================
csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv"
df = pd.read_csv(csv_path, engine="python", encoding="utf-8")

print("Mevcut sütunlar:", df.columns)

if "processed_final_review" in df.columns:
    reviews = df["processed_final_review"].fillna("").tolist()
else:
    reviews = df["final_review"].fillna("").tolist()

sample_review = None
for review in reviews:
    if review.strip():
        sample_review = review.strip()
        break

if not sample_review:
    print("Veride işlenecek bir yorum bulunamadı.")
else:
    print("=== Örnek Yorum ===")
    print(sample_review)
    print("\n====================\n")

# ================================
# Dependency Parsing (Spacy ile)
# ================================
print(">>> Spacy ile Dependency Parsing:")

nlp_spacy = spacy.load("en_core_web_sm")
doc_spacy = nlp_spacy(sample_review)

for token in doc_spacy:
    print(f"{token.text:15} {token.dep_:10} {token.head.text}")

displacy.render(doc_spacy, style="dep", jupyter=True)

# ================================
# Constituency Parsing (Stanza ile) ve Render (Graphviz kullanarak)
# ================================
print("\n>>> Stanza ile Constituency Parsing:")

stanza.download('en', processors='tokenize,pos,constituency', verbose=False)
nlp_stanza = stanza.Pipeline('en', processors='tokenize,pos,constituency', verbose=False)
doc_stanza = nlp_stanza(sample_review)

# Bu fonksiyon NLTK Tree nesnesini Graphviz Digraph formatına çevirir.
def tree_to_graphviz(tree):
    dot = Digraph()
    counter = [0]  # Her node için benzersiz bir ID oluşturmak amacıyla

    def add_nodes_edges(tree, parent_id=None):
        node_id = str(counter[0])
        counter[0] += 1
        # Eğer tree bir Tree nesnesiyse label'ı tree.label() ile, değilse direkt string olarak alırız.
        label = tree.label() if isinstance(tree, Tree) else str(tree)
        dot.node(node_id, label)
        if parent_id is not None:
            dot.edge(parent_id, node_id)
        # Eğer bu node'nun alt elemanları varsa
        if isinstance(tree, Tree):
            for child in tree:
                add_nodes_edges(child, node_id)

    add_nodes_edges(tree)
    return dot

# Her cümlenin constituency ağacını render ediyoruz.
for i, sentence in enumerate(doc_stanza.sentences):
    print(f"\nCümle {i+1} Constituency Ağacı:")
    # Eğer sentence.constituency bir string ise Tree.fromstring ile çeviriyoruz.
    if isinstance(sentence.constituency, str):
        tree = Tree.fromstring(sentence.constituency)
    else:
        tree = sentence.constituency

    # Konsolda metin olarak yazdırıyoruz.
    tree.pretty_print()

    # Graphviz kullanarak görsel render edelim.
    dot = tree_to_graphviz(tree)
    display(dot)


Mevcut sütunlar: Index(['hotel_name', 'author_name', 'language', 'review_text',
       'processed_review', 'processed_final_review'],
      dtype='object')
=== Örnek Yorum ===
stay cheap narrow room know needed room could spend night look hotel cheap room enough room really small bed really small except mini fridge want open large suitcase not open bathroom toilet enough room give night provide with basic condition stay bed pillow comfortable toilet bathroom clean stay night want stay second night room could catch eye night tolerate internet not work room wifi password request location term walk distance everywhere


>>> Spacy ile Dependency Parsing:
stay            ROOT       stay
cheap           amod       room
narrow          amod       room
room            compound   room
know            nmod       room
needed          amod       room
room            nsubj      spend
could           aux        spend
spend           ccomp      stay
night           dobj       spend
look            xc


>>> Stanza ile Constituency Parsing:

Cümle 1 Constituency Ağacı:


ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH